# Model 3 — Binary Logistic Regression dengan Harga Relatif (HargaDiff)
## Pemilihan Hotel: Novotel (pasca-merger) vs Pullman

Model 2 (dengan interaksi Income x TujuanMenginap) sudah signifikan lebih baik dari Model 1,
dan lolos Hosmer-Lemeshow. Tapi di Model 2, **kedua variabel harga (`Harga_Novotel` dan
`Harga_Pullman`) masing-masing TIDAK signifikan sendiri-sendiri** — padahal koefisiennya
hampir sama besar dan berlawanan tanda (Harga_Novotel positif, Harga_Pullman negatif).

Pola ini adalah ciri khas kasus di mana yang sebenarnya relevan bukan level harga
masing-masing brand, melainkan **selisih harga relatif** antar keduanya — logika standar
dalam model pilihan diskrit (discrete choice): konsumen membandingkan harga Pullman
**relatif terhadap** harga Novotel, bukan menilai masing-masing harga secara terpisah.

Model 3 menguji ide ini dengan mengganti `Harga_Novotel` dan `Harga_Pullman` dengan satu
variabel: `HargaDiff = Harga_Pullman - Harga_Novotel`.


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split
from logit_diagnostics import full_diagnostic_report, validation_report, train_fit_report, train_test_comparison


## 1. Load Data & Bangun Variabel HargaDiff

In [2]:
df = pd.read_csv('Data_Binary_Logit_Pullman-vs-Novotel.csv')

df['BrandPilihan_Pullman'] = (df['BrandPilihan'] == 'Pullman').astype(int)
df['TujuanMenginap_Bisnis'] = (df['TujuanMenginap'] == 'Bisnis').astype(int)
df['Income_x_Bisnis'] = df['Income_JutaRp'] * df['TujuanMenginap_Bisnis']

# Variabel baru: selisih harga relatif (positif = Pullman lebih mahal drpd Novotel)
df['HargaDiff'] = df['Harga_Pullman'] - df['Harga_Novotel']

feature_cols_m1 = ['TujuanMenginap_Bisnis', 'Income_JutaRp', 'Harga_Novotel',
                    'Harga_Pullman', 'Iklan_Novotel', 'Iklan_Pullman']
feature_cols_m2 = feature_cols_m1 + ['Income_x_Bisnis']
feature_cols_m3 = ['TujuanMenginap_Bisnis', 'Income_JutaRp', 'HargaDiff',
                   'Iklan_Novotel', 'Iklan_Pullman', 'Income_x_Bisnis']

y = df['BrandPilihan_Pullman']
X1 = df[feature_cols_m1]
X2 = df[feature_cols_m2]
X3 = df[feature_cols_m3]

df[['Harga_Novotel', 'Harga_Pullman', 'HargaDiff']].describe()


,Harga_Novotel,Harga_Pullman,HargaDiff
count,5.000000e+02,5.000000e+02,5.000000e+02
mean,8.513360e+05,1.594392e+06,7.430560e+05
std,1.440095e+05,2.266422e+05,2.678006e+05
min,6.000000e+05,1.204000e+06,1.510000e+05
25%,7.230000e+05,1.389750e+06,5.387500e+05
50%,8.565000e+05,1.594500e+06,7.380000e+05
75%,9.750000e+05,1.778250e+06,9.565000e+05
max,1.100000e+06,1.997000e+06,1.351000e+06


## 2. Split Data (Identik dengan Model 1 & 2)

`random_state=42` yang sama dipakai lagi supaya baris training/testing identik dan
perbandingan ketiga model apple-to-apple.


In [3]:
X1_train, X1_test, y_train, y_test = train_test_split(X1, y, test_size=0.2, random_state=42, stratify=y)
X2_train, X2_test, _, _ = train_test_split(X2, y, test_size=0.2, random_state=42, stratify=y)
X3_train, X3_test, _, _ = train_test_split(X3, y, test_size=0.2, random_state=42, stratify=y)

print('Train:', X3_train.shape, ' | Test:', X3_test.shape)


Train: (400, 6)  | Test: (100, 6)


## 3. Estimasi Model 1, Model 2 (Ulang, sbg Pembanding), dan Model 3

In [4]:
model1 = sm.Logit(y_train, sm.add_constant(X1_train)).fit(disp=0)
model2 = sm.Logit(y_train, sm.add_constant(X2_train)).fit(disp=0)
model3 = sm.Logit(y_train, sm.add_constant(X3_train)).fit(disp=0)

for name, m in [('Model 1', model1), ('Model 2', model2), ('Model 3', model3)]:
    print(f'{name}: LL={m.llf:.4f} | AIC={m.aic:.2f} | BIC={m.bic:.2f} | McFadden R2={m.prsquared:.4f} | k={int(m.df_model)+1}')


Model 1: LL=-203.7967 | AIC=421.59 | BIC=449.53 | McFadden R2=0.2648 | k=7
Model 2: LL=-166.1577 | AIC=348.32 | BIC=380.25 | McFadden R2=0.4006 | k=8
Model 3: LL=-166.6440 | AIC=347.29 | BIC=375.23 | McFadden R2=0.3989 | k=7


## 4. Uji Restriksi: Apakah HargaDiff Sah Menggantikan Dua Harga Terpisah?

Model 3 adalah **restricted version** dari Model 2 — Model 3 memaksa koefisien
`Harga_Novotel` dan `Harga_Pullman` untuk sama besar namun berlawanan tanda
(b_Novotel = -b_Pullman), yang secara matematis sama dengan memasukkan `HargaDiff` sebagai
satu variabel tunggal. Ini adalah **1 restriksi linier**, jadi bisa diuji dengan LR test
antar-model (df=1):

H0: b_Novotel = -b_Pullman (restriksi valid, Model 3 cukup)
H1: restriksi tidak valid, Model 2 (harga terpisah) diperlukan


In [5]:
lr_stat_23 = 2 * (model2.llf - model3.llf)
df_diff_23 = model2.df_model - model3.df_model
lr_pvalue_23 = 1 - stats.chi2.cdf(lr_stat_23, df_diff_23)

print(f'LR statistic (Model 2 vs Model 3) : {lr_stat_23:.4f}')
print(f'df                                : {int(df_diff_23)}')
print(f'p-value                           : {lr_pvalue_23:.4f}')

if lr_pvalue_23 > 0.05:
    print('\n-> GAGAL TOLAK H0: restriksi valid secara statistik.')
    print('   Model 3 (HargaDiff) sama baiknya dengan Model 2, tapi lebih ringkas (1 parameter lebih sedikit).')
else:
    print('\n-> TOLAK H0: kedua harga perlu tetap dipisah (Model 2 lebih tepat).')


LR statistic (Model 2 vs Model 3) : 0.9726
df                                : 1
p-value                           : 0.3240

-> GAGAL TOLAK H0: restriksi valid secara statistik.
   Model 3 (HargaDiff) sama baiknya dengan Model 2, tapi lebih ringkas (1 parameter lebih sedikit).


In [6]:
comparison = pd.DataFrame({
    'Model 1': [model1.llf, model1.aic, model1.bic, model1.prsquared, int(model1.df_model)+1],
    'Model 2': [model2.llf, model2.aic, model2.bic, model2.prsquared, int(model2.df_model)+1],
    'Model 3': [model3.llf, model3.aic, model3.bic, model3.prsquared, int(model3.df_model)+1],
}, index=['Log-Likelihood', 'AIC', 'BIC', 'McFadden R2', 'Jumlah Parameter'])
comparison.round(4)


,Model 1,Model 2,Model 3
Log-Likelihood,-203.7967,-166.1577,-166.6440
AIC,421.5933,348.3154,347.2880
BIC,449.5336,380.2471,375.2283
McFadden R2,0.2648,0.4006,0.3989
Jumlah Parameter,7.0000,8.0000,7.0000


## 5. Full Diagnostic Report — Model 3 (Training)

In [7]:
model3, results3 = full_diagnostic_report(
    X3_train, y_train,
    feature_names=feature_cols_m3,
    y_name='BrandPilihan_Pullman',
    model_label='Model 3 (HargaDiff + Interaction)',
    labels=('Novotel', 'Pullman'),
)



######################################################################
#  FULL DIAGNOSTIC REPORT (Binary Logit) - Model 3 (HargaDiff + Interaction)
######################################################################

1. PEARSON CORRELATION MATRIX (antar variabel independen)
                       TujuanMenginap_Bisnis  Income_JutaRp  HargaDiff  Iklan_Novotel  Iklan_Pullman  Income_x_Bisnis
TujuanMenginap_Bisnis                  1.000          0.050      0.020          0.088          0.037            0.772
Income_JutaRp                          0.050          1.000      0.015          0.043         -0.001            0.470
HargaDiff                              0.020          0.015      1.000          0.110         -0.095            0.006
Iklan_Novotel                          0.088          0.043      0.110          1.000          0.035            0.108
Iklan_Pullman                          0.037         -0.001     -0.095          0.035          1.000            0.047
Income_x_Bisn

## 6. Ringkasan statsmodels (opsional, untuk cross-check)

In [8]:
print(model3.summary())


                            Logit Regression Results                            
Dep. Variable:     BrandPilihan_Pullman   No. Observations:                  400
Model:                            Logit   Df Residuals:                      393
Method:                             MLE   Df Model:                            6
Date:                  Sun, 13 Sep 2026   Pseudo R-squ.:                  0.3989
Time:                          08:30:44   Log-Likelihood:                -166.64
converged:                         True   LL-Null:                       -277.21
Covariance Type:              nonrobust   LLR p-value:                 5.946e-45
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -5.7940      1.005     -5.768      0.000      -7.763      -3.825
TujuanMenginap_Bisnis     7.5010      0.999      7.505      0.000       5.542     

## 7. Bandingkan Signifikansi Variabel Harga: Model 2 vs Model 3

In [9]:
harga_compare = pd.DataFrame({
    'Model 2 (terpisah)': {
        'Harga_Novotel (p-value)': model2.pvalues.get('Harga_Novotel', np.nan),
        'Harga_Pullman (p-value)': model2.pvalues.get('Harga_Pullman', np.nan),
        'HargaDiff (p-value)': np.nan,
    },
    'Model 3 (HargaDiff)': {
        'Harga_Novotel (p-value)': np.nan,
        'Harga_Pullman (p-value)': np.nan,
        'HargaDiff (p-value)': model3.pvalues.get('HargaDiff', np.nan),
    },
})
harga_compare.round(4)


,Model 2 (terpisah),Model 3 (HargaDiff)
Harga_Novotel (p-value),0.0552,NaN
Harga_Pullman (p-value),0.2312,NaN
HargaDiff (p-value),NaN,0.0434


## 8. Fit di Data Training vs Validasi Hold-Out (20%) — Model 3

In [10]:
train_result3 = train_fit_report(model3, X3_train, y_train, label='Training Set (80%) - Model 3',
                                  labels=('Novotel', 'Pullman'))


FIT MODEL DI DATA TRAINING - Training Set (80%) - Model 3
--- Classification Table (threshold = 0.5) ---
                Pred Novotel  Pred Pullman
Actual Novotel           137            60
Actual Pullman            21           182

N                  : 400
Hit Rate (Akurasi) : 0.7975  (79.75%)
Sensitivity (Recall Pullman) : 0.8966
Specificity (Recall Novotel) : 0.6954
Precision (Pullman)          : 0.7521

Proportional Chance Criterion : 0.5001 (hit rate model sebaiknya > 1.25x ini = 0.6251, aturan Hair et al.)
  -> Model MEMENUHI kriteria (>1.25x chance)

--- Press's Q Statistic ---
Press's Q = 141.6100  (nilai kritis chi2(1, 0.05) = 3.8415, p-value = 0.0000)
  -> Signifikan (Q > 3.841): hit rate model JAUH LEBIH BAIK dari tebakan acak



In [11]:
test_result3 = validation_report(model3, X3_test, y_test, label='Hold-Out / Test Set (20%) - Model 3',
                                  labels=('Novotel', 'Pullman'))


VALIDASI MODEL - Hold-Out / Test Set (20%) - Model 3
--- Classification Table (threshold = 0.5) ---
                Pred Novotel  Pred Pullman
Actual Novotel            31            18
Actual Pullman             9            42

N                  : 100
Hit Rate (Akurasi) : 0.7300  (73.00%)
Sensitivity (Recall Pullman) : 0.8235
Specificity (Recall Novotel) : 0.6327
Precision (Pullman)          : 0.7000

Proportional Chance Criterion : 0.5002 (hit rate model sebaiknya > 1.25x ini = 0.6252, aturan Hair et al.)
  -> Model MEMENUHI kriteria (>1.25x chance)

--- Press's Q Statistic ---
Press's Q = 21.1600  (nilai kritis chi2(1, 0.05) = 3.8415, p-value = 0.0000)
  -> Signifikan (Q > 3.841): hit rate model JAUH LEBIH BAIK dari tebakan acak



In [12]:
train_test_comparison(train_result3, test_result3)


RINGKASAN PERBANDINGAN TRAIN vs TEST
Hit Rate         Train      Test         Gap
                0.7975    0.7300     +0.0675  <- indikasi overfitting



## 9. Ringkasan Perbandingan 3 Model & Keputusan Akhir

In [13]:
def hit_rate_holdout(model, X_tr, X_te, y_te):
    Xtr_c = sm.add_constant(X_tr)
    Xte_c = sm.add_constant(X_te, has_constant='add')[Xtr_c.columns]
    pred = model.predict(Xte_c)
    return (((pred >= 0.5).astype(int)) == y_te).mean()

final_summary = pd.DataFrame({
    'Model 1 (sederhana)': {
        'AIC': model1.aic, 'BIC': model1.bic, 'McFadden R2': model1.prsquared,
        'Jumlah Parameter': int(model1.df_model) + 1,
        'Hit Rate Hold-Out': hit_rate_holdout(model1, X1_train, X1_test, y_test),
    },
    'Model 2 (+interaksi)': {
        'AIC': model2.aic, 'BIC': model2.bic, 'McFadden R2': model2.prsquared,
        'Jumlah Parameter': int(model2.df_model) + 1,
        'Hit Rate Hold-Out': hit_rate_holdout(model2, X2_train, X2_test, y_test),
    },
    'Model 3 (+HargaDiff)': {
        'AIC': model3.aic, 'BIC': model3.bic, 'McFadden R2': model3.prsquared,
        'Jumlah Parameter': int(model3.df_model) + 1,
        'Hit Rate Hold-Out': hit_rate_holdout(model3, X3_train, X3_test, y_test),
    },
}).T
final_summary.round(4)


,AIC,BIC,McFadden R2,Jumlah Parameter,Hit Rate Hold-Out
Model 1 (sederhana),421.5933,449.5336,0.2648,7.0,0.72
Model 2 (+interaksi),348.3154,380.2471,0.4006,8.0,0.73
Model 3 (+HargaDiff),347.2880,375.2283,0.3989,7.0,0.73


**Kesimpulan:** Model 3 (HargaDiff) menang tipis dari Model 2 di AIC & BIC (lebih hemat
parameter dengan fit yang secara statistik setara — LR test restriksi tidak signifikan),
sekaligus membuat efek harga relatif menjadi **signifikan** (yang sebelumnya "tersembunyi"
saat dua harga dipisah di Model 2). Hit rate hold-out identik dengan Model 2. Ini menjadikan
Model 3 sebagai **kandidat model final yang lebih parsimoni** dibanding Model 2, dengan
kesimpulan bisnis yang sama namun cerita harga yang lebih jelas: yang menentukan pilihan
brand adalah **seberapa besar Pullman lebih mahal dibanding Novotel**, bukan level harga
masing-masing brand secara terpisah.
